# 🎫 Service Desk Ticket Classifier – Exploration Notebook

Use this notebook to:
- Explore the dataset
- Run quick training experiments
- Visualise results interactively

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.utils import load_config, set_seed, get_device
from src.preprocessing import TextPreprocessor
from src.dataset import DataPipeline
from src.model import build_model
from src.trainer import Trainer
from src.evaluate import evaluate, plot_training_curves
from predict import TicketPredictor

config = load_config('../configs/config.yaml')
set_seed(config['data']['random_seed'])
device = get_device()

## 1. Dataset Overview

In [ ]:
df = pd.read_csv('../data/tickets.csv')
print(f'Total samples: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Class distribution
counts = df['category'].value_counts()
fig, ax = plt.subplots(figsize=(10, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Ticket Category Distribution')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
counts

## 2. Text Length Analysis

In [ ]:
preprocessor = TextPreprocessor(lowercase=True)
df['token_count'] = df['text'].apply(lambda t: len(preprocessor.tokenise(t)))

print(df['token_count'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
df['token_count'].hist(bins=20, ax=ax, color='coral', edgecolor='white')
ax.axvline(df['token_count'].mean(), color='navy', linestyle='--', label='Mean')
ax.set_title('Token Count Distribution')
ax.set_xlabel('Tokens per Ticket')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Run Training

In [ ]:
pipeline = DataPipeline(config)
train_loader, val_loader, test_loader = pipeline.prepare()

model = build_model(
    config['model']['architecture'],
    len(pipeline.vocab),
    len(pipeline.label_encoder),
    config,
)

trainer = Trainer(model, config, pipeline.class_weights, device)
history = trainer.fit(train_loader, val_loader)

In [ ]:
# Plot training curves inline
h = history.to_dict()
epochs = range(1, len(h['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, h['train_loss'], label='Train'); ax1.plot(epochs, h['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(epochs, h['train_acc'], label='Train'); ax2.plot(epochs, h['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Training Curves'); plt.tight_layout(); plt.show()

## 4. Evaluate on Test Set

In [ ]:
metrics = evaluate(
    model        = model,
    loader       = test_loader,
    device       = device,
    class_names  = pipeline.label_encoder.classes,
    results_dir  = '../results/',
    split_name   = 'test_notebook',
)

print(f"\nAccuracy   : {metrics['accuracy']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
print(f"Macro F1   : {metrics['macro_f1']:.4f}")

## 5. Interactive Predictions

In [ ]:
sample_tickets = [
    "My laptop battery drains in 20 minutes even when charging",
    "Cannot log in after returning from holiday leave",
    "The WiFi is super slow on the 4th floor conference room",
    "Need Python and pip installed for the data team",
    "Outlook calendar events disappear after syncing with phone",
]

# Use the trained model directly
import torch, torch.nn.functional as F
from src.preprocessing import encode_and_pad

preprocessor = pipeline.preprocessor
vocab        = pipeline.vocab
le           = pipeline.label_encoder
max_len      = config['preprocessing']['max_length']

model.eval()
for ticket in sample_tickets:
    tokens   = preprocessor.tokenise(ticket)
    seq      = encode_and_pad([tokens], vocab, max_len)
    tensor   = torch.tensor(seq, dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
    pred = le.idx2label[probs.argmax()]
    conf = probs.max()
    print(f"[{pred:<25}  {conf:.1%}]  {ticket}")